# Project II: Economic Growth 

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.linear_model import Lasso

## Read data 

In [ ]:
dat = pd.read_csv('growth.csv')
lbldf = pd.read_csv('labels.csv', index_col='variable')
lbl_all = lbldf.label.to_dict()
print(f'The data contains {dat.shape[0]} rows (countries) and {dat.shape[1]} columns (variables).')

In [ ]:
# convenient to keep a column of ones in the dataset
dat['constant'] = np.ones((dat.shape[0],))

# Selection of controls

This section reduces the candidate controls before the post-double-selection lasso. No convergence regression is estimated here. Each step adds the variables it drops to `vv_dropped`; `dat` itself is not changed.

## 1. Load and inspect

`dat` and `lbl_all` are used as loaded above. The header of `growth.csv` lists `pd1500` twice, and pandas names the second copy `pd1500.1`. If the copy equals `pd1500`, it carries no extra information and is dropped.

In [ ]:
# 1. pandas reads the second pd1500 column as pd1500.1
print(f"pd1500 and pd1500.1 are equal: {dat['pd1500'].equals(dat['pd1500.1'])}")

# 2. drop the copy
vv_dropped = {'pd1500.1': 'load'}

## 2. Trim the sample

The convergence regression needs both `gdp_growth` and `lgdp_initial`, so all later steps use only the countries where both are observed (`dat_trim`).

In [ ]:
# 1. countries with both outcome and regressor
I = dat[['gdp_growth', 'lgdp_initial']].notnull().all(axis=1)

# 2. trimmed sample without the dropped copy
dat_trim = dat.loc[I, [v for v in dat.columns if v not in vv_dropped]]
print(f'Countries before trimming: {dat.shape[0]}, after trimming: {dat_trim.shape[0]}')

## 3. Solow core variables

One variable per Solow concept is protected: it is never dropped in the steps below. `lgdp_initial` is protected as the regressor of interest.

In [ ]:
# 1. one variable per Solow concept
vv_solow = {'Population growth': 'pop_growth',
            'Saving / interest rate': 'investment_rate',
            'Technology growth (proxy)': 'lh_bl',
            'Human capital': 'ls_bl'}

# 2. protected variables are never dropped
protected = ['lgdp_initial'] + list(vv_solow.values())

# 3. non-missing values in the trimmed sample
pd.DataFrame({'concept': list(vv_solow.keys()),
              'chosen': list(vv_solow.values()),
              'non-missing': dat_trim[list(vv_solow.values())].notnull().sum().values})

| Concept | Candidates in the data | Why the chosen variable |
|---|---|---|
| Population growth | `pop_growth`, `population_initial`, `population_now` | `pop_growth` is the growth rate itself; the two levels are only its start and end points. |
| Saving / interest rate | `investment_rate`, `ginv`, `capital_growth_pct_gdp_initial`, `capital_growth_pct_gdp_now` (no interest rate in the data) | `investment_rate` is the investment share of GDP averaged over the whole growth period, the measure named in the assignment. The other investment measures are not protected and go through the steps below like any other control. |
| Technology growth | `lh_bl`; institutions: `polity`, `polity2`, `dem`, `demBMR`, `demCGV`, `marketref`, `currentinst`, `legor_uk`, `legor_fr`; history: `ln_yst`, `pd1500` | No direct measure exists. `lh_bl` (proxy): a workforce with tertiary education is what develops and adopts new technology. |
| Human capital | `ls_bl`, `lp_bl`, `lh_bl`, `leb95`, `imr95` | `ls_bl`: secondary education is the broadest skill measure not already used as the technology proxy; `leb95` and `imr95` are health measures from 1995. |


## 4. Mechanical identities and leakage

Variables linked by definition carry the same information. The data come from different sources, so the relations are applied by definition and are not tested on the data.

Rule: drop the non-protected variable with the most missing values in the trimmed sample. On a tie, keep the log for a log and its level; otherwise drop the derived variable or, for a set that sums to a constant, the base category already used in the notebook. Variables that construct the outcome or the regressor are always dropped (leakage). Numbers in brackets are missing values in the trimmed sample.

### Size variables in logs

Country size affects growth proportionally: doubling the population or the area of a country is expected to change growth by the same amount whether the country is small or large, which is a linear effect of the log, not of the level. In levels, a few very large countries dominate the variation in size, so a handful of countries would decide how size relates to growth. The size measures therefore enter in logs: population in 1970 (`lpop_initial`, already in the data), population in 1, 1000 and 1500 CE, land area, arable land area and oil reserves.

The table below gives the smallest value and the number of zeros of each level in the trimmed sample. `area`, `area_ar`, `pop1`, `pop1000` and `pop1500` are strictly positive and enter as `np.log(x)`. `oilres` is zero for 52 countries and enters as `np.log(1 + x)`, which is zero for a country without oil. The logs are new columns in `dat_trim` (`dat` is not changed). Each log and its level then form a relation in the table of relations, where the tie rule keeps the log.

In [ ]:
# 1. strictly positive size variables: log(x)
vv_size = ['area', 'area_ar', 'pop1', 'pop1000', 'pop1500']
for v in vv_size:
    dat_trim['l' + v] = np.log(dat_trim[v])
    lbl_all['l' + v] = lbl_all[v] + ' (log)'

# 2. oil reserves contain zeros: log(1 + x)
dat_trim['loilres'] = np.log(1 + dat_trim['oilres'])
lbl_all['loilres'] = 'Oil reserves (log of 1 + x)'

# 3. smallest value and number of zeros of each level in the trimmed sample
pd.DataFrame({'min': dat_trim[vv_size + ['oilres']].min(),
              'zeros': (dat_trim[vv_size + ['oilres']] == 0).sum()}).round(3)

In [ ]:
# 1. relations by definition: relation, linked variables, dropped variables
vv_rel = [
    ['leakage: construct the regressor', ['gdp_pc_initial', 'gdp_initial'], ['gdp_pc_initial', 'gdp_initial']],
    ['leakage: construct the outcome', ['gdp_pc_now', 'gdp_now'], ['gdp_pc_now', 'gdp_now']],
    ['lpop_initial = ln(population_initial)', ['lpop_initial', 'population_initial'], ['population_initial']],
    ['larea = ln(area)', ['larea', 'area'], ['area']],
    ['larea_ar = ln(area_ar)', ['larea_ar', 'area_ar'], ['area_ar']],
    ['lpop1 = ln(pop1)', ['lpop1', 'pop1'], ['pop1']],
    ['lpop1000 = ln(pop1000)', ['lpop1000', 'pop1000'], ['pop1000']],
    ['lpop1500 = ln(pop1500)', ['lpop1500', 'pop1500'], ['pop1500']],
    ['loilres = ln(1 + oilres)', ['loilres', 'oilres'], ['oilres']],
    ['pop_growth from population_initial and population_now',
     ['pop_growth', 'population_initial', 'population_now'], ['population_now']],
    ['mortality, its log and exp(log)', ['mortality', 'logem4', 'imputedmort'], ['mortality', 'imputedmort']],
    ['pd1 = pop1 / area', ['pd1', 'pop1', 'area'], ['pd1']],
    ['pd1000 = pop1000 / area', ['pd1000', 'pop1000', 'area'], ['pd1000']],
    ['pd1500 = pop1500 / area', ['pd1500', 'pop1500', 'area'], ['pd1500']],
    ['continent dummies sum to 1', ['africa', 'asia', 'oceania', 'americas', 'europe'], ['europe']],
    ['religion shares sum to 100', ['pprotest', 'pcatholic', 'pmuslim', 'pother'], ['pother']],
    ['Barro-Lee education shares', ['lp_bl', 'ls_bl', 'lh_bl'], ['lp_bl']],
    ['polity2 = polity recoded', ['polity', 'polity2'], ['polity']],
    ['distcr from distc and distr', ['distcr', 'distc', 'distr'], ['distcr']],
]

# 2. add the dropped variables to vv_dropped
for rel, vv, drop in vv_rel:
    for v in drop:
        vv_dropped[v] = 'leakage' if rel.startswith('leakage') else 'identity'

# 3. assert that non-missing polity2 values are within the -10 to 10 scale
assert dat_trim['polity2'].dropna().between(-10, 10).all(), \
    "polity2 contains non-missing values outside the -10 to 10 scale"

# 4. table with missing values in the trimmed sample in brackets
n_miss = dat_trim.isnull().sum()
pd.DataFrame({
    'relation': [r[0] for r in vv_rel],
    'variables': [', '.join(f'{v} ({n_miss[v]})' for v in r[1]) for r in vv_rel],
    'dropped': [', '.join(r[2]) for r in vv_rel]
})

- **Leakage (regressor):** `gdp_pc_initial` is `lgdp_initial` before taking logs, and `gdp_initial` is `gdp_pc_initial` times population. Both are dropped.
- **Leakage (outcome):** `gdp_pc_now` and `gdp_pc_initial` together give `gdp_growth`, and `gdp_now` is `gdp_pc_now` times population. Both are dropped.
- **`lpop_initial` = ln(`population_initial`):** neither is missing. For a log and its level the log is kept, so `population_initial` is dropped and population in 1970 enters in logs.
- **Size variables and their logs:** each log has the same missing values as its level, so the tie rule drops the levels `area`, `area_ar`, `pop1`, `pop1000`, `pop1500` and `oilres`.
- **`pop_growth`** is the annual growth rate from `population_initial` to `population_now`. `pop_growth` is protected and none of the three is missing. Given population in 1970 (kept as `lpop_initial`) and `pop_growth`, `population_now` is derived and is dropped.
- **`mortality`, `logem4`, `imputedmort`** are one mortality measure in levels, logs and exp(log). `mortality` (51 missing) and `imputedmort` (35) are dropped, and `logem4` (31) is kept.
- **Population density = population / `area`:** in 1, 1000 and 1500 CE, density and population have the same number of missing values (17, 7 and 3), so the derived densities are dropped. `area` has none.
- **Continent dummies** sum to one and none is missing, so the base category `europe` (as in `vv_geography`) is dropped.
- **Religion shares** sum to 100 and none is missing, so the base category `pother` (as in `vv_religion`) is dropped.
- **Barro-Lee shares** split the population by education level. `ls_bl` and `lh_bl` are protected, so `lp_bl` (as in `vv_educ`) is dropped.
- **`polity2`** is `polity` recoded. Both have 16 missing values, but `polity2` corrects the invalid `-88` codes for PAK and SDN. Therefore, `polity` is dropped and `polity2` is retained for the analysis.
- **`distcr`** (distance to coast or river) is a function of `distc` and `distr`. All three have 9 missing, so the derived `distcr` is dropped.

## 5. Correlation pruning

Two controls with an absolute correlation of 0.95 or more carry almost the same information. Pairs are taken from the highest correlation down. In each pair, the variable with fewer missing values is kept and protected variables are never dropped. Ties are decided by economic interpretation, and pairs that contain an already dropped variable are skipped. `lgdp_initial` is included so that no control can be a near-copy of the regressor. Numbers in brackets are missing values in the trimmed sample.

In [ ]:
# 1. remaining candidates, including the regressor and excluding the column of ones
vv_cand = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth', 'constant']]

# 2. absolute correlations above the diagonal, highest first
C = dat_trim[vv_cand].corr().abs()
upper = np.triu(np.ones(C.shape, dtype=bool), k=1)
pairs = C.where(upper).stack().sort_values(ascending=False)

# 3. round half up to two decimals and keep pairs of 0.95 or more
pairs = (pairs + 1e-9).round(2)
pairs = pairs[pairs >= 0.95]

# 4. drop the variable with more missing values; ties are decided by hand
vv_tie = {('demCGV', 'demBMR'): 'demCGV', ('dem', 'demBMR'): 'demBMR'}
rows = []
for (a, b), r in pairs.items():
    if a in vv_dropped or b in vv_dropped or (a in protected and b in protected):
        drop = ''
    elif a in protected or b in protected:
        drop = b if a in protected else a
    elif n_miss[a] != n_miss[b]:
        drop = a if n_miss[a] > n_miss[b] else b
    else:
        drop = vv_tie[(a, b)]
    if drop:
        vv_dropped[drop] = 'correlation'
    rows.append([f'{a} ({n_miss[a]})', f'{b} ({n_miss[b]})', r, drop])

pd.DataFrame(rows, columns=['a', 'b', 'corr', 'dropped'])

Drops and ties (missing values in brackets):

- `lpop1000` (7) and `lpop1` (17) are dropped against `lpop1500` (3): log populations in different centuries are near-copies of each other. In logs, `lpop1500` is not a near-copy of population in 1970 (`lpop_initial`), so both are kept.
- `pdivhmi` (24) and `pdivhmi_aa` (25) are dropped against `pdiv` (0) and `pdiv_aa` (5). They are alternative versions of predicted genetic diversity.
- `capital_growth_pct_gdp_initial` (17) is dropped against `ginv` (16): both measure investment as a share of GDP. `capital_growth_pct_gdp_initial` is not protected, so the variable with fewer missing values is kept.
- `abslat` (1) is dropped against `uvdamage` (0). UV exposure is largely determined by distance from the equator.
- **Tie, `demCGV` and `demBMR` (12 each).** Both are 0/1 codings of democracy, but `demCGV` also requires that power has changed hands, so a stable democracy where one party keeps winning is coded as non-democratic. `demBMR` is easier to interpret and is kept.
- **Tie, `dem` and `demBMR` (12 each).** `dem` is the source's own democracy measure (ANRR) and `demBMR` an alternative coding of the same concept, so `dem` is kept.
- The pair `lpop1` and `lpop1000` involves a variable that was already dropped and needs no action.

## 6. Missingness

The lasso needs every control observed for every country. Controls missing for more than `MAX_MISSING_SHARE` of the trimmed sample are dropped (protected controls never are), and then only complete cases are kept.

In [ ]:
# 1. share of missing values per remaining control in the trimmed sample
MAX_MISSING_SHARE = 0.20
vv_x = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth', 'lgdp_initial', 'constant']]
miss = dat_trim[vv_x].isnull().mean()

# 2. drop controls above the threshold, never protected ones
for v in vv_x:
    if miss[v] > MAX_MISSING_SHARE and v not in protected:
        vv_dropped[v] = 'missingness'
print(f"Dropped (missing share above {MAX_MISSING_SHARE}): {[v for v in vv_x if v in vv_dropped]}")

# 3. table of missing shares
pd.DataFrame({'missing share': miss,
              'protected': [v in protected for v in vv_x],
              'dropped': [v in vv_dropped for v in vv_x]}).sort_values('missing share', ascending=False).round(3)

In [ ]:
# 1. remaining controls
vv_x = [v for v in vv_x if v not in vv_dropped]

# 2. keep complete cases on outcome, regressor and controls
J = dat_trim[['gdp_growth', 'lgdp_initial'] + vv_x].notnull().all(axis=1)
dat_pre = dat_trim.loc[J, ['code', 'gdp_growth', 'lgdp_initial'] + vv_x]
print(f'Countries in the trimmed sample: {dat_trim.shape[0]}, complete cases: {dat_pre.shape[0]}')

In [ ]:
# 1. countries lost and the variables they miss
L = dat_trim.loc[~J, ['code'] + vv_x]
pd.DataFrame({'code': L['code'],
              'n missing': L[vv_x].isnull().sum(axis=1),
              'missing': L[vv_x].isnull().apply(lambda r: ', '.join(r.index[r]), axis=1)}).sort_values(['n missing', 'code'])

## 7. Kept versus dropped

The table below compares the countries in the final sample (kept) with the other countries in the trimmed sample (dropped) for the outcome, the regressor and the Solow variables: the mean of each in the two groups and the number of dropped countries that observe it. Every kept country observes every variable, which the code checks. The table is written to `output/kept_vs_dropped.tex` with a caption. Countries without `gdp_growth` or `lgdp_initial` can never enter the estimation sample, so they are left out, and the two groups together make up the trimmed sample.

In [ ]:
# 1. countries in the final sample (kept) and the other countries in the trimmed sample (dropped)
K = dat['code'].isin(dat_pre['code'])
D = dat['code'].isin(dat_trim['code']) & ~K

# 2. means in both groups and the number of dropped countries that observe each variable
vv = ['gdp_growth', 'lgdp_initial'] + list(vv_solow.values())
assert dat.loc[K, vv].notnull().all().all(), "a kept country misses a variable"
tab = pd.DataFrame({'Mean in kept countries': dat.loc[K, vv].mean(),
                    'Mean in dropped countries': dat.loc[D, vv].mean(),
                    'Countries dropped': dat.loc[D, vv].count()})

# 3. write the table to LaTeX with a caption
caption = 'Kept versus dropped countries'
note = (r'Mean of each variable for countries in the final sample (kept) and the other countries in the trimmed sample, '
        r'which observe both $g_i$ and $y_{i0}$ (dropped), and the number of dropped countries that observe the variable; '
        f'all {K.sum()} kept countries observe every variable.')
lines = [r'\begin{table}[!htbp]', r'\centering',
         r'\caption{' + caption + '}', r'\label{tab:kept_vs_dropped}',
         r'\begin{threeparttable}', r'\begin{tabular}{lrrr}', r'\toprule',
         r' & \multicolumn{1}{l}{Mean in} & \multicolumn{1}{l}{Mean in} & \multicolumn{1}{l}{Countries} \\',
         r' & \multicolumn{1}{l}{kept countries} & \multicolumn{1}{l}{dropped countries} & \multicolumn{1}{l}{dropped} \\',
         r'\midrule']
lines += [r'\texttt{' + v.replace('_', r'\_') + '} & ' + f'{mk:.3f} & {md:.3f} & {n}' + r' \\'
          for v, mk, md, n in zip(vv, tab.iloc[:, 0], tab.iloc[:, 1], tab.iloc[:, 2])]
lines += [r'\bottomrule', r'\end{tabular}',
          r'\begin{tablenotes}[flushleft]', r'\footnotesize',
          r'\item \textit{Note:} ' + note,
          r'\item \textit{Source:} Own calculations based on provided growth data.',
          r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
Path('output').mkdir(exist_ok=True)
Path('output/kept_vs_dropped.tex').write_text('\n'.join(lines))
tab.style.set_caption(caption).format(precision=3)

Kept and dropped countries grew at the same rate (mean `gdp_growth` 0.016 in both groups), so the restrictions do not select on the outcome. Dropped countries started somewhat poorer (mean `lgdp_initial` 7.967 against 7.490).

- **Similar.** Population growth (0.017 against 0.018) and the average investment rate (`investment_rate` 22.588 against 23.748) are close in the two groups.
- **Different.** Kept countries have a higher secondary education share (`ls_bl` 12.750 against 8.007) and more than twice the tertiary share (`lh_bl` 3.126 against 1.511).
- **Coverage.** Only 21 of the 35 dropped countries observe the education shares and 32 observe `investment_rate`, so those means rest on fewer countries.

Countries are dropped for missing controls, and the dropped countries are less educated and somewhat poorer, so the estimate of $\beta$ describes a final sample that is more educated than the trimmed sample as a whole. The comparison covers observed characteristics only and says nothing about differences in unobservables.

## 8. Summary

Final counts and the final control names in `vv_prescreened`, which the estimation uses together with `dat_pre`. The table below lists every variable: the used ones (the regressor `lgdp_initial` and the controls) and the dropped ones, together with the step that dropped them. The used variables with their labels and the ISO codes of the countries in the final sample are written to `output/used_variables.tex` and `output/countries.tex`.

In [ ]:
# final counts
vv_prescreened = [v for v in dat_pre.columns if v not in ['code', 'gdp_growth', 'lgdp_initial']]
print(f'Countries (rows): {dat_pre.shape[0]}')
print(f'Explanatory variables (columns minus gdp_growth and code): {dat_pre.shape[1] - 2}')
print(f'Controls excluding lgdp_initial: {len(vv_prescreened)}')

In [ ]:
# 1. used variables (regressor and controls) and dropped variables with the step that dropped them
vv_used = ['lgdp_initial'] + vv_prescreened
pd.DataFrame({'variable': vv_used + list(vv_dropped),
              'status': ['used'] * len(vv_used) + ['dropped'] * len(vv_dropped),
              'step': ['regressor'] + ['control'] * len(vv_prescreened) + list(vv_dropped.values()),
              'label': [lbl_all.get(v, '') for v in vv_used + list(vv_dropped)]})

In [ ]:
# 1. LaTeX special characters in labels
tex = lambda s: s.replace('%', r'\%').replace('&', r'\&').replace('_', r'\_').replace('#', r'\#').replace('$', r'\$')

# 2. used variables with their labels
tab_used = pd.DataFrame({'Variable': vv_used,
                         'Label': [lbl_all[v][:1].upper() + lbl_all[v][1:] for v in vv_used]})

# 3. two pairs of columns, filled column by column; labels wrap within their column
R = -(-len(tab_used) // 2)
pairs = [r'\texttt{' + tex(v) + r'} & \raggedright ' + tex(l) for v, l in tab_used.values] + [' & '] * (2 * R - len(tab_used))
rows = [' & '.join(pairs[r::R]) + r' \tabularnewline' for r in range(R)]

# 4. write the table to LaTeX with a caption
note = (r'The regressor $y_{i0}$ and the ' + str(len(vv_prescreened)) + r' candidate controls in $\mathbf{z}_i$ left after the selection of controls. '
        r'Labels from the provided data; the logs of the size variables are constructed in the selection of controls.')
lines = [r'\begin{table}[!htbp]', r'\centering',
         r'\caption{Variables in the estimation}', r'\label{tab:used_variables}',
         r'\begin{threeparttable}', r'\begin{tabular}{lp{0.28\linewidth}lp{0.28\linewidth}}', r'\toprule',
         r'Variable & \raggedright Label & Variable & \raggedright Label \tabularnewline', r'\midrule']
lines += rows
lines += [r'\bottomrule', r'\end{tabular}',
          r'\begin{tablenotes}[flushleft]', r'\footnotesize',
          r'\item \textit{Note:} ' + note,
          r'\item \textit{Source:} Own calculations based on provided growth data.',
          r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
Path('output').mkdir(exist_ok=True)
Path('output/used_variables.tex').write_text('\n'.join(lines))
tab_used

In [ ]:
# 1. ISO codes of the countries in the final sample in ten columns, filled column by column
codes = sorted(dat_pre['code'])
R = -(-len(codes) // 10)
codes_pad = codes + [''] * (10 * R - len(codes))
tab_ctry = pd.DataFrame([codes_pad[r::R] for r in range(R)])

# 2. write the table to LaTeX with a caption
note = f'ISO codes of the {len(codes)} countries in the final sample.'
lines = [r'\begin{table}[!htbp]', r'\centering',
         r'\caption{Countries in the final sample}', r'\label{tab:countries}',
         r'\begin{threeparttable}', r'\begin{tabular}{' + 'l' * 10 + '}', r'\toprule']
lines += [' & '.join(row) + r' \\' for row in tab_ctry.values]
lines += [r'\bottomrule', r'\end{tabular}',
          r'\begin{tablenotes}[flushleft]', r'\footnotesize',
          r'\item \textit{Note:} ' + note,
          r'\item \textit{Source:} Own calculations based on provided growth data.',
          r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
Path('output').mkdir(exist_ok=True)
Path('output/countries.tex').write_text('\n'.join(lines))
tab_ctry

# Estimation

Post double Lasso and post partialling out Lasso of `gdp_growth` (in percent per year) on `lgdp_initial`, with the prescreened controls `vv_prescreened` as candidates. Every Lasso step uses the BCCH penalty ($c = 1.1$, $\alpha = 0.05$) on standardized regressors.

In [ ]:
def standardize(X):
    return (X - np.mean(X, axis=0))/np.std(X, axis=0, ddof=1)

def BCCH(X_tilde, y, c=1.1, alpha=0.05):
    (N, p) = X_tilde.shape

    # pilot penalty from deviations from the mean
    # sklearn's Lasso minimizes (1/2N)*RSS + alpha*|b|_1, so its alpha is half the penalty in the report
    yXscale = (np.max((X_tilde.T ** 2) @ ((y - np.mean(y)) ** 2) / N)) ** 0.5
    penalty_pilot = c / np.sqrt(N) * stats.norm.ppf(1-alpha/(2*p)) * yXscale

    # updated penalty from the pilot residuals
    pred = Lasso(alpha=penalty_pilot, max_iter=10000).fit(X_tilde, y).predict(X_tilde)
    eps = y - pred
    epsXscale = (np.max((X_tilde.T ** 2) @ (eps ** 2) / N)) ** 0.5
    return c / np.sqrt(N) * stats.norm.ppf(1-alpha/(2*p)) * epsXscale

def fmt(x, k=3):
    return f'{x:.{k}f}'.replace('-', '$-$')

def stars(b, se):
    p = 2*(1 - stats.norm.cdf(abs(b/se)))
    return '$^{***}$' if p < 0.01 else '$^{**}$' if p < 0.05 else '$^{*}$' if p < 0.1 else ''

def latex_table(caption, label, b_label, b, se, ci_low, ci_high, rows_sel, N, p, note):
    lines = [r'\begin{table}[!htbp]', r'\centering',
             r'\caption{' + caption + '}', r'\label{' + label + '}',
             r'\begin{threeparttable}', r'\begin{tabular}{lc}', r'\toprule',
             r' & Annual growth in GDP p.c. (\%) \\', r'\midrule',
             b_label + ' & ' + fmt(b) + stars(b, se) + r' \\',
             r' & (' + fmt(se) + r') \\', r'\addlinespace',
             r'95\% confidence interval & [' + fmt(ci_low) + ', ' + fmt(ci_high) + r'] \\', r'\midrule']
    lines += [k + ' & ' + v + r' \\' for k, v in rows_sel]
    lines += [r'\midrule',
              r'Observations (countries) & ' + str(N) + r' \\',
              r'Candidate controls & ' + str(p) + r' \\',
              r'\bottomrule', r'\end{tabular}',
              r'\begin{tablenotes}[flushleft]', r'\footnotesize',
              r'\item \textit{Note:} ' + note,
              r'\item \textit{Source:} Own calculations based on provided growth data.',
              r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
    return '\n'.join(lines)

## Post double Lasso

In [ ]:
# 1. data
y = dat_pre['gdp_growth'].values * 100.0
d = dat_pre['lgdp_initial'].values
Z = dat_pre[vv_prescreened].values
X = np.column_stack((d, Z))
N = X.shape[0]

X_stan = standardize(X)
Z_stan = standardize(Z)
d_stan = standardize(d)

# 2. Lasso of growth on initial income and the controls
penalty_BCCHyx = BCCH(X_stan, y)
fit_BCCHyx = Lasso(penalty_BCCHyx, max_iter=10000).fit(X_stan, y)
coefs_yx = fit_BCCHyx.coef_
resyx = y - fit_BCCHyx.predict(X_stan)
resyxz = resyx + d_stan*coefs_yx[0] # growth minus the part explained by the controls

# 3. Lasso of initial income on the controls
penalty_BCCHdz = BCCH(Z_stan, d)
fit_BCCHdz = Lasso(penalty_BCCHdz, max_iter=10000).fit(Z_stan, d)
coefs_dz = fit_BCCHdz.coef_
resdz = d - fit_BCCHdz.predict(Z_stan)

# 4. estimate, robust standard error and 95% confidence interval
beta_PDL = (resdz@resyxz)/(resdz@d)
sigma2_PDL = (resdz**2@resyx**2/N)/((resdz@resdz/N)**2)
se_PDL = np.sqrt(sigma2_PDL/N)
q = stats.norm.ppf(1-0.025)
CI_low_PDL  = beta_PDL - q*se_PDL
CI_high_PDL = beta_PDL + q*se_PDL

# 5. selected controls
vv_sel_yx = [z for z, b in zip(vv_prescreened, coefs_yx[1:]) if b != 0]
vv_sel_dz = [z for z, b in zip(vv_prescreened, coefs_dz) if b != 0]
print(f'N = {N}, candidate controls = {len(vv_prescreened)}')
print(f'lambda_BCCH (Y on D, Z): {penalty_BCCHyx:.4f}, selected controls: {vv_sel_yx}')
print(f'lambda_BCCH (D on Z):    {penalty_BCCHdz:.4f}, selected controls: {vv_sel_dz}')

# 6. LaTeX table
tex_PDL = latex_table(
    caption=r'Post double Lasso estimate of $\beta$',
    label='tab:pdl_estimates',
    b_label=r'Log GDP p.c. in 1970, $\check{\beta}$',
    b=beta_PDL, se=se_PDL, ci_low=CI_low_PDL, ci_high=CI_high_PDL,
    rows_sel=[(r'Penalty $\hat{\lambda}$, growth on initial GDP and controls', fmt(penalty_BCCHyx)),
              (r'Penalty $\hat{\lambda}$, initial GDP on controls', fmt(penalty_BCCHdz)),
              ('Selected controls, growth equation', str(len(vv_sel_yx))),
              ('Selected controls, initial GDP equation', str(len(vv_sel_dz)))],
    N=N, p=len(vv_prescreened),
    note=r'Post double Lasso estimate of $\beta$ in $g_i = \beta y_{i0} + \mathbf{z}_i\boldsymbol{\gamma} + u_i$. '
         r'BCCH penalties ($c=1.1$, $\alpha=0.05$) on standardized regressors. '
         r'Heteroskedasticity-robust standard errors in parentheses. '
         r'$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$, for tests against zero.')
Path('output').mkdir(exist_ok=True)
Path('output/pdl_estimates.tex').write_text(tex_PDL)

pd.DataFrame({'Estimate of beta': beta_PDL, 'Standard error': se_PDL,
              'Low bound of CI': CI_low_PDL, 'High bound of CI': CI_high_PDL}, index=['PDL']).round(4)

## Post partialling out Lasso

In [ ]:
# 1. Lasso of growth on the controls (the Lasso of initial income on the controls, resdz, is the one above)
penalty_BCCHyz = BCCH(Z_stan, y)
fit_BCCHyz = Lasso(penalty_BCCHyz, max_iter=10000).fit(Z_stan, y)
coefs_yz = fit_BCCHyz.coef_
resyz = y - fit_BCCHyz.predict(Z_stan)

# 2. estimate, robust standard error and 95% confidence interval
beta_PPOL = (resdz@resyz)/(resdz@resdz)
sigma2_PPOL = (resyz**2@resdz**2/N)/((resdz@resdz/N)**2)
se_PPOL = np.sqrt(sigma2_PPOL/N)
CI_low_PPOL  = beta_PPOL - q*se_PPOL
CI_high_PPOL = beta_PPOL + q*se_PPOL

# 3. selected controls
vv_sel_yz = [z for z, b in zip(vv_prescreened, coefs_yz) if b != 0]
print(f'lambda_BCCH (Y on Z): {penalty_BCCHyz:.4f}, selected controls: {vv_sel_yz}')
print(f'lambda_BCCH (D on Z): {penalty_BCCHdz:.4f}, selected controls: {vv_sel_dz}')

# 4. LaTeX table
tex_PPOL = latex_table(
    caption=r'Post partialling out Lasso estimate of $\beta$',
    label='tab:ppol_estimates',
    b_label=r'Log GDP p.c. in 1970, $\breve{\beta}$',
    b=beta_PPOL, se=se_PPOL, ci_low=CI_low_PPOL, ci_high=CI_high_PPOL,
    rows_sel=[(r'Penalty $\hat{\lambda}$, growth on controls', fmt(penalty_BCCHyz)),
              (r'Penalty $\hat{\lambda}$, initial GDP on controls', fmt(penalty_BCCHdz)),
              ('Selected controls, growth equation', str(len(vv_sel_yz))),
              ('Selected controls, initial GDP equation', str(len(vv_sel_dz)))],
    N=N, p=len(vv_prescreened),
    note=r'Post partialling out Lasso estimate of $\beta$ in $g_i = \beta y_{i0} + \mathbf{z}_i\boldsymbol{\gamma} + u_i$. '
         r'BCCH penalties ($c=1.1$, $\alpha=0.05$) on standardized regressors. '
         r'Heteroskedasticity-robust standard errors in parentheses. '
         r'$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$, for tests against zero.')
Path('output').mkdir(exist_ok=True)
Path('output/ppol_estimates.tex').write_text(tex_PPOL)

# 5. both estimates
results = {'PDL' : [beta_PDL,  se_PDL,  CI_low_PDL,  CI_high_PDL],
           'PPOL': [beta_PPOL, se_PPOL, CI_low_PPOL, CI_high_PPOL]}
pd.DataFrame.from_dict(results, orient='index',
                       columns=['Estimate of beta', 'Standard error', 'Low bound of CI', 'High bound of CI']).round(4)

## Both estimates in one table

The post double Lasso and post partialling out Lasso estimates side by side, written to `output/pdl_ppol_est.tex` with a caption. A dash marks a penalty that the method does not use.

In [ ]:
# 1. rows with one entry per method; '--' where the method has no such penalty
rows = [(r'Penalty $\hat{\lambda}$, growth on initial GDP and controls', fmt(penalty_BCCHyx), '--'),
        (r'Penalty $\hat{\lambda}$, growth on controls', '--', fmt(penalty_BCCHyz)),
        (r'Penalty $\hat{\lambda}$, initial GDP on controls', fmt(penalty_BCCHdz), fmt(penalty_BCCHdz)),
        ('Selected controls, growth equation', str(len(vv_sel_yx)), str(len(vv_sel_yz))),
        ('Selected controls, initial GDP equation', str(len(vv_sel_dz)), str(len(vv_sel_dz)))]

# 2. LaTeX table with both estimates
note = (r'Post double Lasso ($\check{\beta}$) and post partialling out Lasso ($\breve{\beta}$) estimates of $\beta$ in $g_i = \beta y_{i0} + \mathbf{z}_i\boldsymbol{\gamma} + u_i$. '
        r'BCCH penalties ($c=1.1$, $\alpha=0.05$) on standardized regressors. '
        r'Heteroskedasticity-robust standard errors in parentheses. '
        r'$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$, for tests against zero.')
lines = [r'\begin{table}[!htbp]', r'\centering',
         r'\caption{Post double Lasso and post partialling out Lasso estimates of $\beta$}', r'\label{tab:pdl_ppol_est}',
         r'\begin{threeparttable}', r'\begin{tabular}{lcc}', r'\toprule',
         r' & \multicolumn{2}{c}{Annual growth in GDP p.c. (\%)} \\', r'\cmidrule(lr){2-3}',
         r' & Post double Lasso & Post partialling out Lasso \\',
         r' & $\check{\beta}$ & $\breve{\beta}$ \\', r'\midrule',
         r'Log GDP p.c. in 1970 & ' + fmt(beta_PDL) + stars(beta_PDL, se_PDL) + ' & ' + fmt(beta_PPOL) + stars(beta_PPOL, se_PPOL) + r' \\',
         r' & (' + fmt(se_PDL) + ') & (' + fmt(se_PPOL) + r') \\', r'\addlinespace',
         r'95\% confidence interval & [' + fmt(CI_low_PDL) + ', ' + fmt(CI_high_PDL) + '] & ['
         + fmt(CI_low_PPOL) + ', ' + fmt(CI_high_PPOL) + r'] \\', r'\midrule']
lines += [k + ' & ' + a + ' & ' + b + r' \\' for k, a, b in rows]
lines += [r'\midrule',
          r'Observations (countries) & ' + str(N) + ' & ' + str(N) + r' \\',
          r'Candidate controls & ' + str(len(vv_prescreened)) + ' & ' + str(len(vv_prescreened)) + r' \\',
          r'\bottomrule', r'\end{tabular}',
          r'\begin{tablenotes}[flushleft]', r'\footnotesize',
          r'\item \textit{Note:} ' + note,
          r'\item \textit{Source:} Own calculations based on provided growth data.',
          r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
Path('output').mkdir(exist_ok=True)
Path('output/pdl_ppol_est.tex').write_text('\n'.join(lines));